# Reinício controlado da simulação

> ⚠️ **Operação destrutiva.** Apaga todo o estado da simulação e da bronze para que a simulação recomece do primeiro dia (`2026-09-06`). Não faz parte da operação normal e nunca deve ser agendado em job.

**O que é apagado, nesta ordem** (desenho A7):

| # | Item | Por que nesta posição |
|---|---|---|
| 1 | tabelas `olist.bronze.<entidade>` | consumidoras primeiro: nada fica lendo um estado pela metade |
| 2 | checkpoints da bronze (`ops.checkpoints/bronze/`) | **sempre junto com as tabelas**: checkpoint sem tabela, ou tabela sem checkpoint, causa perda ou duplicidade (B3) |
| 3 | arquivos da landing (`raw_data/<entidade>/` e `_staging/`) | |
| 4 | registros de `ops.simulator_runs` e `ops.simulator_chaos_log` | por último: o progresso volta ao início só depois que o resto foi apagado |

**O que é mantido:** catálogo, schemas, volumes, a fonte `olist_source` e a estrutura das tabelas de controle.

**Como usar:** rode o notebook com o widget `confirmacao` vazio para ver o que será apagado. Para apagar de fato, preencha `confirmacao` com `APAGAR` e rode de novo. Depois, **limpe o widget**.

In [0]:
# O QUE FAZ: cria o parâmetro de confirmação. Vazio (padrão), o notebook só mostra o que seria
#   apagado; com "APAGAR", executa a exclusão.
#
# FUNÇÕES:
# - dbutils.widgets.text(nome, valor_padrão, rótulo): cria um widget de texto (se já existe,
#   nada muda). O valor é lido com dbutils.widgets.get(nome) e sempre chega como string.
dbutils.widgets.text("confirmacao", "", "Digite APAGAR para confirmar")

In [0]:
# O QUE FAZ: define exatamente o que pode ser apagado. As listas são explícitas de propósito:
#   o reset apaga as 10 entidades conhecidas, e não "tudo o que existir no schema". Uma tabela
#   criada no futuro em olist.bronze não é apagada por engano.
#
# RECURSOS:
# - tuple (...): sequência que não pode ser alterada depois de criada.
# - f"...": f-string; todos os caminhos e nomes derivam de CATALOG.
CATALOG = "olist"
CONFIRMATION_WORD = "APAGAR"

RAW_DIR = f"/Volumes/{CATALOG}/landing/raw_data"
STAGING_DIR = f"{RAW_DIR}/_staging"
CHECKPOINT_VOLUME = f"/Volumes/{CATALOG}/ops/checkpoints"
BRONZE_CHECKPOINT_DIR = f"{CHECKPOINT_VOLUME}/bronze"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
CONTROL_TABLES = (f"{CATALOG}.ops.simulator_runs", f"{CATALOG}.ops.simulator_chaos_log")

ENTITIES = (
    "orders",
    "order_status_events",
    "order_items",
    "order_payments",
    "order_reviews",
    "customers",
    "sellers",
    "products",
    "geolocation",
    "category_translation",
)

In [0]:
# O QUE FAZ: declara as funções de inventário e mostra o estado atual: tabelas bronze,
#   checkpoints, pastas da landing e linhas das tabelas de controle. É o "antes" do reset e,
#   com o widget vazio, o único efeito do notebook.
#
# FUNÇÕES DECLARADAS:
# - list_names(caminho): nomes das pastas e arquivos dentro do caminho. Só é chamada em raízes
#   de volume (RAW_DIR, CHECKPOINT_VOLUME), que sempre existem; a pasta bronze/ dos checkpoints
#   só é listada se aparecer na raiz, então nenhuma chamada depende de exceção.
#   Usa dbutils.fs, a mesma API da exclusão: os.path.exists (acesso via FUSE) guarda um cache
#   e continuou vendo as pastas logo depois do dbutils.fs.rm.
# - inventory(): devolve um dicionário com o que existe em cada um dos 4 grupos do reset.
#
# FUNÇÕES NATIVAS:
# - dbutils.fs.ls(caminho): lista o conteúdo; cada item tem .name (pastas terminam com "/").
# - str.rstrip("/"): remove a barra final do nome da pasta.
# - spark.catalog.tableExists(nome): True se a tabela existe no catálogo.
# - spark.table(nome).count(): número de linhas da tabela.
# - {x for x in ...}: set comprehension; [x for x in ... if ...]: list comprehension com filtro.
def list_names(path: str) -> set[str]:
    """Nomes das pastas e arquivos dentro de path (sem a barra final)."""
    return {f.name.rstrip("/") for f in dbutils.fs.ls(path)}


def inventory() -> dict[str, list[str]]:
    """O que existe hoje em cada grupo que o reset apaga."""
    checkpoints = list_names(BRONZE_CHECKPOINT_DIR) if "bronze" in list_names(CHECKPOINT_VOLUME) else set()
    landing = list_names(RAW_DIR)
    return {
        "1. tabelas bronze": [
            f"{BRONZE_SCHEMA}.{e}" for e in ENTITIES if spark.catalog.tableExists(f"{BRONZE_SCHEMA}.{e}")
        ],
        "2. checkpoints bronze": [f"{BRONZE_CHECKPOINT_DIR}/{e}" for e in ENTITIES if e in checkpoints],
        "3. landing": [f"{RAW_DIR}/{n}" for n in (*ENTITIES, "_staging") if n in landing],
        "4. tabelas de controle": [
            f"{t} ({spark.table(t).count()} linhas)" for t in CONTROL_TABLES
        ],
    }


antes = inventory()
for grupo, itens in antes.items():
    print(f"{grupo} ({len(itens)}):")
    for item in itens or ["(nada)"]:
        print(f"    {item}")

In [0]:
# O QUE FAZ: só deixa o notebook seguir para a exclusão se o widget tiver exatamente a palavra
#   de confirmação. Caso contrário, encerra com sucesso sem apagar nada.
#
# FUNÇÕES NATIVAS:
# - dbutils.widgets.get(nome).strip(): valor do widget sem espaços nas pontas.
# - dbutils.notebook.exit(texto): encerra o notebook nesse ponto, com sucesso; as células
#   seguintes não rodam.
if dbutils.widgets.get("confirmacao").strip() != CONFIRMATION_WORD:
    dbutils.notebook.exit(f"Nada foi apagado. Para executar o reset, preencha confirmacao = {CONFIRMATION_WORD}.")

print("Confirmação recebida: executando o reset.")

In [0]:
# O QUE FAZ: apaga os 4 grupos, na ordem do desenho A7. Cada comando é idempotente: apagar algo
#   que já não existe não gera erro, então o reset pode ser repetido se for interrompido.
#
# COMANDOS:
# - DROP TABLE IF EXISTS t: remove a tabela do catálogo. Em tabela gerenciada do Unity Catalog,
#   os dados podem ser recuperados por 7 dias com UNDROP TABLE (rede de segurança contra engano).
# - dbutils.fs.rm(caminho, recurse=True): apaga a pasta e tudo dentro dela.
# - DELETE FROM t: apaga todas as linhas e mantém a tabela, as colunas, os comentários e as
#   constraints (a estrutura criada na etapa 1.2a continua intacta).
for e in ENTITIES:
    spark.sql(f"DROP TABLE IF EXISTS {BRONZE_SCHEMA}.{e}")
print("1. tabelas bronze apagadas")

dbutils.fs.rm(BRONZE_CHECKPOINT_DIR, recurse=True)
print("2. checkpoints bronze apagados")

for path in [f"{RAW_DIR}/{e}" for e in ENTITIES] + [STAGING_DIR]:
    dbutils.fs.rm(path, recurse=True)
print("3. landing apagada")

for table in CONTROL_TABLES:
    spark.sql(f"DELETE FROM {table}")
print("4. tabelas de controle esvaziadas")

In [0]:
# O QUE FAZ: refaz o inventário e confirma que os grupos 1 a 3 ficaram vazios e que as
#   tabelas de controle estão com 0 linhas.
#
# FUNÇÕES NATIVAS:
# - all(...): True se todos os itens forem verdadeiros.
# - str.endswith(texto): o texto termina com...
depois = inventory()
for grupo, itens in depois.items():
    print(f"{grupo}: {itens or '(nada)'}")

reset_ok = (
    not depois["1. tabelas bronze"]
    and not depois["2. checkpoints bronze"]
    and not depois["3. landing"]
    and all(item.endswith("(0 linhas)") for item in depois["4. tabelas de controle"])
)
print("\nReset:", "OK — simulação pronta para recomeçar em 2026-09-06" if reset_ok else "INCOMPLETO")